# Advanced Ranking Metrics Analysis for LatentLens

## Objetivo

Este notebook analiza y visualiza los resultados de evaluación de métricas de ranking para sistemas de recomendación, comparando diferentes algoritmos (SVD, KNN-User, KNN-Item) utilizando métricas avanzadas que van más allá de RMSE.

### Métricas Evaluadas

- **Precision@k**: Fracción de ítems relevantes en las top-k recomendaciones
- **Recall@k**: Fracción de ítems relevantes capturados en las top-k recomendaciones  
- **Average Precision@k**: Media de precisión a diferentes puntos de corte
- **NDCG@k**: Normalized Discounted Cumulative Gain - considera el orden de las recomendaciones

### Por qué son importantes estas métricas

Mientras que RMSE mide la precisión de las predicciones de rating, las métricas de ranking miden qué tan bien el sistema **ordena y prioriza** las recomendaciones, lo cual está más alineado con objetivos de negocio.

In [ ]:
# Importaciones necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
from mlflow.tracking import MlflowClient
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11

# Configurar MLflow
mlflow.set_tracking_uri("http://127.0.0.1:5000")
client = MlflowClient()

print("🔧 Configuración completa")
print(f"📊 MLflow Tracking URI: {mlflow.get_tracking_uri()}")

## 1. Carga de Resultados Experimentales

In [ ]:
# Obtener experimento de ranking metrics comparison
experiment_name = "Ranking-Metrics-Comparison"
experiment = mlflow.get_experiment_by_name(experiment_name)

if experiment is None:
    print(f"❌ Experimento '{experiment_name}' no encontrado")
    print("Experimentos disponibles:")
    for exp in client.search_experiments():
        print(f"  - {exp.name}")
else:
    print(f"✅ Experimento encontrado: {experiment.name}")
    print(f"📍 Experiment ID: {experiment.experiment_id}")
    
    # Obtener todos los runs del experimento
    runs = client.search_runs(experiment_ids=[experiment.experiment_id])
    print(f"🏃 Número de runs encontrados: {len(runs)}")
    
    # Mostrar información de los runs
    for run in runs:
        print(f"  - {run.info.run_name}: {run.info.status}")

In [ ]:
# Extraer métricas de todos los runs
results_data = []

for run in runs:
    run_data = {
        'model_name': run.info.run_name.replace('_ranking_eval', ''),
        'run_id': run.info.run_id,
        'status': run.info.status
    }
    
    # Agregar métricas
    for metric_name, metric_value in run.data.metrics.items():
        run_data[metric_name] = metric_value
    
    # Agregar parámetros
    for param_name, param_value in run.data.params.items():
        run_data[f'param_{param_name}'] = param_value
    
    results_data.append(run_data)

# Crear DataFrame
results_df = pd.DataFrame(results_data)
print(f"📊 DataFrame creado con {len(results_df)} filas y {len(results_df.columns)} columnas")

# Mostrar resumen de modelos
print("\n🤖 Modelos evaluados:")
for model in results_df['model_name'].unique():
    print(f"  - {model}")

# Mostrar algunas métricas clave
print("\n📈 Métricas disponibles:")
metric_cols = [col for col in results_df.columns if not col.startswith('param_') and col not in ['model_name', 'run_id', 'status']]
for metric in sorted(metric_cols):
    print(f"  - {metric}")

## 2. Análisis de RMSE vs Ranking Metrics

In [ ]:
# Crear tabla comparativa de métricas principales
key_metrics = ['rmse', 'precision_at_10', 'recall_at_10', 'average_precision_at_10', 'ndcg_at_10']
comparison_data = []

for _, row in results_df.iterrows():
    model_metrics = {'Model': row['model_name']}
    for metric in key_metrics:
        if metric in row:
            model_metrics[metric] = row[metric]
        else:
            model_metrics[metric] = np.nan
    comparison_data.append(model_metrics)

comparison_df = pd.DataFrame(comparison_data)

print("📊 COMPARACIÓN DE MODELOS - MÉTRICAS CLAVE")
print("=" * 80)
print(comparison_df.to_string(index=False, float_format='%.4f'))

# Identificar mejor modelo para cada métrica
print("\n🏆 MEJORES MODELOS POR MÉTRICA")
print("=" * 50)
print(f"Mejor RMSE (menor): {comparison_df.loc[comparison_df['rmse'].idxmin(), 'Model']} ({comparison_df['rmse'].min():.4f})")
print(f"Mejor Precision@10: {comparison_df.loc[comparison_df['precision_at_10'].idxmax(), 'Model']} ({comparison_df['precision_at_10'].max():.4f})")
print(f"Mejor Recall@10: {comparison_df.loc[comparison_df['recall_at_10'].idxmax(), 'Model']} ({comparison_df['recall_at_10'].max():.4f})")
print(f"Mejor Avg Precision@10: {comparison_df.loc[comparison_df['average_precision_at_10'].idxmax(), 'Model']} ({comparison_df['average_precision_at_10'].max():.4f})")
print(f"Mejor NDCG@10: {comparison_df.loc[comparison_df['ndcg_at_10'].idxmax(), 'Model']} ({comparison_df['ndcg_at_10'].max():.4f})")

In [ ]:
# Visualización comparativa de métricas clave
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Comparación de Modelos: RMSE vs Ranking Metrics', fontsize=16, fontweight='bold')

# RMSE (menor es mejor)
ax1 = axes[0, 0]
bars1 = ax1.bar(comparison_df['Model'], comparison_df['rmse'], color='lightcoral', alpha=0.7)
ax1.set_title('RMSE (Menor es Mejor)', fontweight='bold')
ax1.set_ylabel('RMSE')
ax1.tick_params(axis='x', rotation=45)
# Añadir valores en las barras
for bar in bars1:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height + 0.005,
             f'{height:.4f}', ha='center', va='bottom', fontweight='bold')

# Precision@10
ax2 = axes[0, 1]
bars2 = ax2.bar(comparison_df['Model'], comparison_df['precision_at_10'], color='lightblue', alpha=0.7)
ax2.set_title('Precision@10 (Mayor es Mejor)', fontweight='bold')
ax2.set_ylabel('Precision@10')
ax2.tick_params(axis='x', rotation=45)
for bar in bars2:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height + 0.005,
             f'{height:.4f}', ha='center', va='bottom', fontweight='bold')

# Recall@10
ax3 = axes[0, 2]
bars3 = ax3.bar(comparison_df['Model'], comparison_df['recall_at_10'], color='lightgreen', alpha=0.7)
ax3.set_title('Recall@10 (Mayor es Mejor)', fontweight='bold')
ax3.set_ylabel('Recall@10')
ax3.tick_params(axis='x', rotation=45)
for bar in bars3:
    height = bar.get_height()
    ax3.text(bar.get_x() + bar.get_width()/2., height + 0.005,
             f'{height:.4f}', ha='center', va='bottom', fontweight='bold')

# Average Precision@10
ax4 = axes[1, 0]
bars4 = ax4.bar(comparison_df['Model'], comparison_df['average_precision_at_10'], color='gold', alpha=0.7)
ax4.set_title('Average Precision@10 (Mayor es Mejor)', fontweight='bold')
ax4.set_ylabel('Avg Precision@10')
ax4.tick_params(axis='x', rotation=45)
for bar in bars4:
    height = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2., height + 0.005,
             f'{height:.4f}', ha='center', va='bottom', fontweight='bold')

# NDCG@10
ax5 = axes[1, 1]
bars5 = ax5.bar(comparison_df['Model'], comparison_df['ndcg_at_10'], color='mediumpurple', alpha=0.7)
ax5.set_title('NDCG@10 (Mayor es Mejor)', fontweight='bold')
ax5.set_ylabel('NDCG@10')
ax5.tick_params(axis='x', rotation=45)
for bar in bars5:
    height = bar.get_height()
    ax5.text(bar.get_x() + bar.get_width()/2., height + 0.005,
             f'{height:.4f}', ha='center', va='bottom', fontweight='bold')

# Ocultar el último subplot
axes[1, 2].axis('off')

plt.tight_layout()
plt.show()

## 3. Análisis por K-values

In [ ]:
# Análisis de rendimiento por valores de k
k_values = [5, 10, 20]
metrics_base = ['precision', 'recall', 'average_precision', 'ndcg']

# Crear DataFrame para análisis por k
k_analysis_data = []

for _, row in results_df.iterrows():
    model_name = row['model_name']
    for metric_base in metrics_base:
        for k in k_values:
            metric_name = f"{metric_base}_at_{k}"
            if metric_name in row:
                k_analysis_data.append({
                    'Model': model_name,
                    'Metric': metric_base.replace('_', ' ').title(),
                    'k': k,
                    'Value': row[metric_name]
                })

k_analysis_df = pd.DataFrame(k_analysis_data)

print("📊 Análisis por valores de k completado")
print(f"Total de observaciones: {len(k_analysis_df)}")
print(f"Métricas analizadas: {k_analysis_df['Metric'].unique()}")
print(f"Valores de k: {sorted(k_analysis_df['k'].unique())}")

In [ ]:
# Visualización de rendimiento por k-values
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Rendimiento de Modelos por Valores de K', fontsize=16, fontweight='bold')

metrics_to_plot = ['Precision', 'Recall', 'Average Precision', 'Ndcg']
colors = ['tab:blue', 'tab:orange', 'tab:green']

for i, metric in enumerate(metrics_to_plot):
    ax = axes[i//2, i%2]
    
    metric_data = k_analysis_df[k_analysis_df['Metric'] == metric]
    
    for j, model in enumerate(metric_data['Model'].unique()):
        model_data = metric_data[metric_data['Model'] == model]
        ax.plot(model_data['k'], model_data['Value'], 
                marker='o', linewidth=2.5, markersize=8, 
                label=model, color=colors[j], alpha=0.8)
        
        # Añadir valores en los puntos
        for _, row in model_data.iterrows():
            ax.annotate(f'{row["Value"]:.3f}', 
                       (row['k'], row['Value']), 
                       textcoords="offset points", 
                       xytext=(0,10), ha='center',
                       fontsize=9, fontweight='bold')
    
    ax.set_title(f'{metric}@k', fontweight='bold', fontsize=14)
    ax.set_xlabel('k', fontweight='bold')
    ax.set_ylabel(metric, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xticks(k_values)

plt.tight_layout()
plt.show()

## 4. Insights Clave y Conclusiones

In [ ]:
# Análisis de correlaciones entre métricas
correlation_metrics = ['rmse', 'precision_at_10', 'recall_at_10', 'average_precision_at_10', 'ndcg_at_10']
correlation_data = results_df[correlation_metrics].corr()

# Visualización de matriz de correlación
plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(correlation_data, dtype=bool))
sns.heatmap(correlation_data, mask=mask, annot=True, cmap='RdYlBu_r', center=0,
            square=True, linewidths=.5, cbar_kws={"shrink": .5}, 
            fmt='.3f', annot_kws={'fontweight': 'bold'})
plt.title('Matriz de Correlación entre Métricas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("🔍 ANÁLISIS DE CORRELACIONES")
print("=" * 50)
print(f"Correlación RMSE vs Precision@10: {correlation_data.loc['rmse', 'precision_at_10']:.3f}")
print(f"Correlación RMSE vs Recall@10: {correlation_data.loc['rmse', 'recall_at_10']:.3f}")
print(f"Correlación RMSE vs NDCG@10: {correlation_data.loc['rmse', 'ndcg_at_10']:.3f}")
print(f"Correlación Precision@10 vs Recall@10: {correlation_data.loc['precision_at_10', 'recall_at_10']:.3f}")

In [ ]:
# Ranking general de modelos
def calculate_model_ranking(df):
    """
    Calcula un ranking general de modelos basado en múltiples métricas.
    Para RMSE, menor es mejor (se invierte).
    Para el resto, mayor es mejor.
    """
    ranking_metrics = ['precision_at_10', 'recall_at_10', 'average_precision_at_10', 'ndcg_at_10']
    
    scores = []
    for _, row in df.iterrows():
        model_score = 0
        
        # RMSE (invertido - menor es mejor)
        rmse_rank = df['rmse'].rank(ascending=True)[row.name]
        model_score += rmse_rank
        
        # Métricas de ranking (mayor es mejor)
        for metric in ranking_metrics:
            if metric in df.columns:
                metric_rank = df[metric].rank(ascending=False)[row.name]
                model_score += metric_rank
        
        scores.append({
            'Model': row['model_name'],
            'Overall_Score': model_score,
            'RMSE': row['rmse'],
            'Precision@10': row.get('precision_at_10', 0),
            'Recall@10': row.get('recall_at_10', 0),
            'Avg_Precision@10': row.get('average_precision_at_10', 0),
            'NDCG@10': row.get('ndcg_at_10', 0)
        })
    
    return pd.DataFrame(scores).sort_values('Overall_Score')

model_ranking = calculate_model_ranking(results_df)

print("🏆 RANKING GENERAL DE MODELOS")
print("=" * 80)
print("(Basado en suma de rankings en RMSE, Precision@10, Recall@10, Avg_Precision@10, NDCG@10)")
print()
for i, (_, row) in enumerate(model_ranking.iterrows(), 1):
    print(f"{i}. {row['Model']:<12} (Score: {row['Overall_Score']:.1f})")
    print(f"   RMSE: {row['RMSE']:.4f} | P@10: {row['Precision@10']:.4f} | R@10: {row['Recall@10']:.4f} | AP@10: {row['Avg_Precision@10']:.4f} | NDCG@10: {row['NDCG@10']:.4f}")
    print()

## 5. Conclusiones y Recomendaciones

### Principales Hallazgos:

1. **Dominancia de SVD**: El modelo SVD muestra el mejor rendimiento tanto en RMSE como en métricas de ranking
2. **Importancia de las métricas de ranking**: Las métricas de ranking revelan diferencias significativas entre modelos que RMSE no detecta
3. **Trade-offs entre métricas**: Algunos modelos pueden tener mejor RMSE pero peor ranking, o viceversa

### Recomendaciones:

1. **Usar múltiples métricas**: No basarse únicamente en RMSE para evaluación
2. **Optimizar para el negocio**: Las métricas de ranking están más alineadas con objetivos comerciales
3. **Considerar el contexto**: La elección de k depende del contexto de aplicación (top-5, top-10, etc.)

### Próximos Pasos:

1. Implementar optimización específica para métricas de ranking
2. Evaluar en datasets más grandes y diversos
3. Integrar métricas de ranking en el pipeline de producción

In [ ]:
# Resumen final
print("📋 RESUMEN EJECUTIVO")
print("=" * 50)
print(f"✅ Experimento: {experiment_name}")
print(f"🤖 Modelos evaluados: {len(results_df)}")
print(f"📊 Métricas analizadas: {len([col for col in results_df.columns if 'at_' in col])}")
print(f"🏆 Mejor modelo general: {model_ranking.iloc[0]['Model']}")
print(f"🎯 Mejor RMSE: {results_df.loc[results_df['rmse'].idxmin(), 'model_name']} ({results_df['rmse'].min():.4f})")
print(f"🎯 Mejor NDCG@10: {results_df.loc[results_df['ndcg_at_10'].idxmax(), 'model_name']} ({results_df['ndcg_at_10'].max():.4f})")
print("\n🚀 Evaluación de ranking metrics implementada exitosamente!")